In [1]:
import pyspark.sql.functions as F

from pyspark.sql import SparkSession
import os
from pyspark.sql.types import *
from pyspark.sql.functions import col, year, month, when

In [2]:
spark = SparkSession \
    .builder \
    .config("spark.streaming.stopGracefullyOnShutdown", True) \
    .config("spark.sql.shuffle.partitions", 4) \
    .master("local[*]") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/12 13:11:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Data Streaming

In [3]:
#Creating a Data Schema from one of the files
static_df = (
    spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .option("ignoreTrailingWhiteSpace", "true")
        .option("ignoreLeadingWhiteSpace", "true")
        .option("mode", "DROPMALFORMED")
        .load("Data/report_2014_1.csv")
)

schema = static_df.schema

In [4]:
#Defining the streaming

# Path to the directory containing the CSV files
input_path = "Data"

#Files processed per trigger
NUM_FILES_PER_TRIGGER = 1

# Read the streaming DataFrame from the directory
streaming_df = spark.readStream \
    .option("maxFilesPerTrigger", NUM_FILES_PER_TRIGGER) \
    .option("header", "true") \
    .format("csv") \
    .schema(schema) \
    .option("ignoreTrailingWhiteSpace", "true") \
    .option("ignoreLeadingWhiteSpace", "true") \
    .option("mode", "DROPMALFORMED") \
    .load(input_path)


In [5]:
#Data cleaning process

column_mapping = {
    "FL_DATE": ("FlightDate", DateType()),
    "OP_CARRIER": ("Reporting_Airline", StringType()),
    "OP_CARRIER_FL_NUM": ("Flight_Number_Reporting_Airline", IntegerType()),
    "ORIGIN_CITY": ("OriginCityName", StringType()),
    "ORIGIN": ("Origin", StringType()),
    "DEST_CITY": ("DestCityName", StringType()),
    "DEST": ("Dest", StringType()),
    "CRS_DEP_TIME": ("CRSDepTime", IntegerType()),
    "DEP_TIME": ("DepTime", FloatType()),
    "DEP_DELAY": ("DepDelay", FloatType()),
    "TAXI_OUT": ("TaxiOut", FloatType()),
    "WHEELS_OFF": ("WheelsOff", FloatType()),
    "WHEELS_ON": ("WheelsOn", FloatType()),
    "TAXI_IN": ("TaxiIn", FloatType()),
    "CRS_ARR_TIME": ("CRSArrTime", IntegerType()),
    "ARR_TIME": ("ArrTime", FloatType()),
    "ARR_DELAY": ("ArrDelay", FloatType()),
    "CANCELLED": ("Cancelled", FloatType()),
    "CANCELLATION_CODE": ("CancellationCode", StringType()),
    "DIVERTED": ("Diverted", FloatType()),
    "CRS_ELAPSED_TIME": ("CRSElapsedTime", FloatType()),
    "ACTUAL_ELAPSED_TIME": ("ActualElapsedTime", FloatType()),
    "AIR_TIME": ("AirTime", FloatType()),
    "DISTANCE": ("Distance", FloatType()),
    "CARRIER_DELAY": ("CarrierDelay", FloatType()),
    "WEATHER_DELAY": ("WeatherDelay", FloatType()),
    "NAS_DELAY": ("NASDelay", FloatType()),
    "SECURITY_DELAY": ("SecurityDelay", FloatType()),
    "LATE_AIRCRAFT_DELAY": ("LateAircraftDelay", FloatType())
}

selected_cols = [
    col(src).cast(dtype).alias(dst) for dst, (src, dtype) in column_mapping.items()
]

streaming_df = streaming_df.select(*selected_cols)

#Fill nulls in delay columns
delay_cols = [
    "ARR_DELAY", "DEP_DELAY", "CARRIER_DELAY", "WEATHER_DELAY",
    "NAS_DELAY", "SECURITY_DELAY", "LATE_AIRCRAFT_DELAY"
]
streaming_df = streaming_df.na.fill(0, subset=delay_cols)

#Add Year and Month columns
streaming_df = streaming_df.withColumn("Year", year(col("FL_DATE"))) \
                           .withColumn("Month", month(col("FL_DATE")))

#Add binary delayed flag
streaming_df = streaming_df.withColumn("IsDelayed",when(col("ARR_DELAY") > 15, 1).otherwise(0))

#Drop cancelled flights
streaming_df = streaming_df.filter(col("CANCELLED") == 0).drop("CANCELLATION_CODE")

In [6]:
#Starting the process of collecting dt, cleaning and then storing it
#Output data is in "output" folder and the new data is added every 10 sec
cleaning = (
    streaming_df.writeStream
        .trigger(processingTime="10 seconds")
        .option("maxFilesPerTrigger", NUM_FILES_PER_TRIGGER)
        .option("path", "output/flights")
        .option("checkpointLocation", "checkpoint/flights")
        .format("parquet")
        .outputMode("append")
        .partitionBy("Year", "Month")
        .start()
)

25/12/12 13:12:04 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [15]:
#Testing with some simple queries done at different times while the streaming is still in work
df_try = spark.read.parquet("output/flights/Year=*/Month=*")
df_try.groupBy("ORIGIN", "DEST").count().sort("count",ascending = False).show()

+------+----+-----+
|ORIGIN|DEST|count|
+------+----+-----+
|   SFO| LAX|23812|
|   LAX| SFO|23401|
|   JFK| LAX|20404|
|   LAX| JFK|20393|
|   LAS| LAX|18035|
|   LAX| LAS|17925|
|   LGA| ORD|15328|
|   ORD| LGA|15195|
|   HNL| OGG|14628|
|   SEA| LAX|14627|
|   OGG| HNL|14612|
|   LAX| SEA|14590|
|   ATL| MCO|14239|
|   MCO| ATL|14231|
|   LAX| ORD|13585|
|   SFO| LAS|13338|
|   ATL| LGA|13336|
|   LGA| ATL|13287|
|   ORD| LAX|13193|
|   SFO| JFK|13187|
+------+----+-----+
only showing top 20 rows



In [21]:
#This was executed a couple of triggers later
df_try = spark.read.parquet("output/flights/Year=*/Month=*")
df_try.groupBy("ORIGIN", "DEST").count().sort("count",ascending = False).show()

+------+----+-----+
|ORIGIN|DEST|count|
+------+----+-----+
|   SFO| LAX|68499|
|   LAX| SFO|67479|
|   LAX| JFK|59234|
|   JFK| LAX|59233|
|   LAS| LAX|52750|
|   LAX| LAS|52419|
|   LGA| ORD|49782|
|   ORD| LGA|49535|
|   HNL| OGG|45025|
|   OGG| HNL|44991|
|   ATL| MCO|41599|
|   MCO| ATL|41555|
|   SEA| LAX|41536|
|   LAX| SEA|41491|
|   LAX| ORD|39899|
|   ATL| LGA|39730|
|   LGA| ATL|39606|
|   ORD| LAX|38849|
|   SFO| JFK|38682|
|   JFK| SFO|38608|
+------+----+-----+
only showing top 20 rows



In [22]:
#!!!!!!!!!!!!!!!!!!!!Run this once you are finished, not immidiately!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
cleaning.stop()

In [25]:
df_streamed = spark.read.parquet("output/flights/Year=*/Month=*")
#The number of rows total
print(df_streamed.count())

27621193
